In [ ]:
import os
import glob
import pandas as pd
import json
import re
from pathlib import Path
from typing import Dict, List, Any


FLOW_SOURCE_DIR = "../../storage/akvo-flow"
# Mapping files are in the forms subdirectory
MAPPINGS_DIR = "./output/forms"
# MIS form JSONs are in the original forms directory
FORMS_SOURCE_DIR = "../../backend/source/forms"

# Column names for question mapping
FLOW_QUESTION_ID_COL = "flow_question_id"
MIS_QUESTION_ID_COL = "mis_question_id"
MIS_FORM_ID_COL = "mis_form_id"
MIS_QUESTION_LABEL_COL = "mis_question_label"
MIS_QUESTION_ORDER_COL = "mis_question_order"

# Caddisfly constants
CADDISFLY_TYPE = "caddisfly"
CADDISFLY_RESULT = "result"
CADDISFLY_NAME_KEY = "name"
CADDISFLY_UNIT_KEY = "unit"

TYPE_KEY = "type"
VALUE_KEY = "value"

def load_data_file(
    flow_id: int
) -> pd.DataFrame:
    """
    Load data CSV file for the specified Flow form ID.

    Args:
        flow_id: The Akvo Flow form ID

    Returns:
        DataFrame with data or None if file not found/invalid
    """
    csv_path = os.path.join(FLOW_SOURCE_DIR, "raw", f"{flow_id}_*.csv")
    # Find matching file (handles versioning like _v1, _v2)
    try:
        matching_files = glob.glob(csv_path)
        if not matching_files:
            return None
        csv_path = matching_files[0]  # Use first match
        df = pd.read_csv(
            csv_path,
            encoding="utf-8",
            low_memory=False,  # Better for large files
        )
        return df
    except Exception as e:
        print(f"Error loading data file for Flow ID {flow_id}: {e}")
        return None

def load_question_mappings(
    flow_id: int,
    mis_form_id: int = None,
    child_form_ids: List[int] = None
) -> Dict[str, List[str]]:
    """
    Load question mappings from separate parent-child CSV files.
    
    Looks for:
    - {flow_id}_mapping_parent_{mis_id}.csv
    - {flow_id}_mapping_child_{child_id}.csv (only for expected child IDs)

    Args:
        flow_id: The Akvo Flow form ID
        mis_form_id: The MIS form ID (required)
        child_form_ids: Optional list of expected child form IDs to validate against.
                       If provided, only child mapping files matching these IDs will be loaded.

    Returns:
        List of dicts grouped by mis_form_id with question mappings
    """
    if mis_form_id is None:
        raise ValueError("mis_form_id is required")
    
    dfs_to_combine = []
    
    # Load parent mapping file
    parent_mapping_path = os.path.join(
        MAPPINGS_DIR, f"{flow_id}_mapping_parent_{mis_form_id}.csv"
    )
    
    if not os.path.exists(parent_mapping_path):
        raise FileNotFoundError(f"Parent mapping file not found: {parent_mapping_path}")
    
    parent_df = pd.read_csv(
        parent_mapping_path,
        encoding="utf-8",
        dtype={FLOW_QUESTION_ID_COL: str, MIS_QUESTION_ID_COL: str},
    )
    dfs_to_combine.append(parent_df)
    print(f"Loaded parent mapping: {os.path.basename(parent_mapping_path)}")
    
    # Find all child mapping files
    child_mapping_pattern = os.path.join(
        MAPPINGS_DIR, f"{flow_id}_mapping_child_*.csv"
    )
    child_mapping_files = glob.glob(child_mapping_pattern)
    
    # If child_form_ids is provided, validate and filter child mapping files
    if child_form_ids is not None:
        # Build expected filenames
        expected_child_files = {
            os.path.join(MAPPINGS_DIR, f"{flow_id}_mapping_child_{child_id}.csv")
            for child_id in child_form_ids
        }
        
        # Check for expected files that are missing
        for expected_file in expected_child_files:
            if not os.path.exists(expected_file):
                print(f"Warning: Expected child mapping file not found: {os.path.basename(expected_file)}")
        
        # Only load child mapping files that match expected child form IDs
        child_mapping_files = [
            f for f in child_mapping_files
            if f in expected_child_files
        ]
        
        print(f"Validated child mapping files against {len(child_form_ids)} expected child form IDs")
    
    # Load child mapping files
    for child_file in child_mapping_files:
        child_df = pd.read_csv(
            child_file,
            encoding="utf-8",
            dtype={FLOW_QUESTION_ID_COL: str, MIS_QUESTION_ID_COL: str},
        )
        dfs_to_combine.append(child_df)
        print(f"Loaded child mapping: {os.path.basename(child_file)}")
    
    # Combine all mappings
    df = pd.concat(dfs_to_combine, ignore_index=True)
    print(f"Loaded {len(dfs_to_combine)} mapping files for Flow ID {flow_id}")
    
    # Convert mis_question_order to numeric for proper sorting
    df[MIS_QUESTION_ORDER_COL] = pd.to_numeric(df[MIS_QUESTION_ORDER_COL], errors="coerce")
    
    # Group by mis_form_id and return
    df_grouped = df.groupby(MIS_FORM_ID_COL)
    return [
        {
            MIS_FORM_ID_COL: mis_form_id,
            "questions": group_df[[FLOW_QUESTION_ID_COL, MIS_QUESTION_ID_COL, MIS_QUESTION_LABEL_COL, MIS_QUESTION_ORDER_COL]]
            .dropna(subset=[FLOW_QUESTION_ID_COL, MIS_QUESTION_ID_COL])
            .query(
                f"{MIS_QUESTION_ID_COL} != '' and {FLOW_QUESTION_ID_COL} != ''"
            )
            .sort_values(by=MIS_QUESTION_ORDER_COL, ascending=True)
            .apply(
                lambda row: {
                    FLOW_QUESTION_ID_COL: row[FLOW_QUESTION_ID_COL],
                    MIS_QUESTION_ID_COL: row[MIS_QUESTION_ID_COL],
                    MIS_QUESTION_LABEL_COL: row[MIS_QUESTION_LABEL_COL],
                    MIS_QUESTION_ORDER_COL: row[MIS_QUESTION_ORDER_COL],
                },
                axis=1,
            )
            .tolist(),
        }
        for mis_form_id, group_df in df_grouped
    ]

def load_json_form(
    mis_form_id: str
):
    # Find MIS form JSON
    json_path = next(filter(
        lambda f: f.suffix == ".json" and mis_form_id in f.name,
        Path(FORMS_SOURCE_DIR).iterdir(),
    ), None)
    if not json_path:
        return []
    j = pd.read_json(json_path, typ="series").to_dict()
    df = pd.json_normalize(
        j,
        record_path=["question_groups", "questions"],
        meta= [
            "id",
            "form",
            "type",
            "version",
            ["question_groups", "label"],
            ["question_groups", "repeatable"],
        ],
        meta_prefix=f"mis_",
        errors="ignore"
    )
    df = df.rename(
        columns={
            "mis_question_groups.label": "question_group",
            "mis_question_groups.repeatable": "question_group_repeatable"
        }
    )
    return df.to_dict(orient="records")

def normalize_value(value: Any) -> Any:
    """
    Normalize various value types to string format for database storage.
    Args:
        value: The input value to normalize.
    """
    if pd.isna(value):
        return ""

    # If it's a JSON string (common in Flow exports)
    if isinstance(value, str) and (
        value.startswith("{") or value.startswith("[")
    ):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, str):
                # If the parsed value is still a string, normalize it
                return normalize_value(parsed)
            return parsed
        except json.JSONDecodeError:
            return str(value).strip()
    return value

def transform_mis_value(
    questions: List[dict],
    question_id: str,
    answer: str = None
):
    answer = normalize_value(answer)

    # find questions by question_id
    fq = list(filter(lambda q: str(q["id"]) == question_id, questions))
    if len(fq) == 0:
        return answer
    question = fq[0]
    if question["type"] == "administration" and isinstance(answer, list):
        return "|".join([
            str(a["name"]).strip() if "name" in a else str(a["text"]).strip()
            for a in answer
        ])
    if question["type"] == "geo" and isinstance(answer, dict):
        lat = answer.get("lat", "")
        lon = answer.get("long", "")
        return f"{lat}|{lon}"
    if question["type"] in [
        "photo",
        "signature",
    ]:
        if "image" in answer:
            return f"data:image/png;base64,{answer['image']}"
        if "filename" in answer:
            return answer["filename"]
        if isinstance(answer, list):
            return "|".join([
              f"data:image/png;base64,{a['image']}"
                for a in answer
                if "image" in a
            ])
    if isinstance(answer, list):
        val_list = []
        for a in answer:
            if "text" in a:
                val_option = a["text"]
                if (
                    "options" in question and
                    isinstance(question["options"], list) and
                    len(question["options"]) > 0
                ):
                    # find option value by label
                    find_option = next(filter(
                        lambda o: o["label"] == a["text"],
                        question["options"]
                    ), None)
                    if find_option and "value" in find_option:
                        val_option = find_option["value"]
                val_list.append(str(val_option).strip())
            if "isOther" in a and a["isOther"] == True:
                val_list.append("other")
        return "|".join(val_list)
    if isinstance(answer, dict) and "text" in answer:
        val_option = answer["text"]
        if (
            "options" in question and
            isinstance(question["options"], list) and
            len(question["options"]) > 0
        ):
            # find option value by label
            find_option = next(filter(
                lambda o: o["label"] == answer["text"],
                question["options"]
            ), None)
            if find_option and "value" in find_option:
                val_option = find_option["value"]
        return str(val_option).strip()
    if question["type"] == "number":
        try:
            num_value = float(
                re.sub(r"[\s]", "", str(answer))
            )
            return num_value
        except ValueError:
            return answer
    return answer


def build_sorted_column_mapping(
    question_id_to_label: Dict[str, str],
    seed_questions: List[dict]
) -> Dict[str, str]:
    """
    Build a column mapping that sorts columns by mis_question_order.
    
    Args:
        question_id_to_label: Mapping of question IDs to labels
        seed_questions: List of form mappings with questions sorted by order
    
    Returns:
        Sorted column mapping dictionary
    """
    # Build a list of (question_id, label, order) tuples
    column_order = []
    
    for fq in seed_questions:
        for q in fq["questions"]:
            qid = q[MIS_QUESTION_ID_COL]
            qlabel = q.get(MIS_QUESTION_LABEL_COL, qid)
            # Use the actual mis_question_order value from the mapping
            qorder = q.get(MIS_QUESTION_ORDER_COL, float("inf"))
            if pd.notna(qlabel) and qlabel != "":
                column_order.append((qid, qlabel, qorder))
    
    # Sort by mis_question_order (numerical value)
    column_order.sort(key=lambda x: x[2] if pd.notna(x[2]) else float("inf"))
    
    # Build the mapping in sorted order
    sorted_mapping = {}
    for qid, qlabel, _ in column_order:
        sorted_mapping[qid] = qlabel
        sorted_mapping[str(qid)] = qlabel
        # Also add version with .0 suffix for float-converted IDs
        if '.' not in str(qid):
            sorted_mapping[f"{qid}.0"] = qlabel
    
    return sorted_mapping


def build_ordered_column_list(
    seed_questions: List[dict],
    metadata_cols: List[str] = None
) -> List[str]:
    """
    Build an ordered list of column names for DataFrame reordering.
    Uses human-readable labels.
    
    Args:
        seed_questions: List of form mappings with questions sorted by order
        metadata_cols: List of metadata column names to place first (optional)
    
    Returns:
        Ordered list of column names (labels)
    """
    if metadata_cols is None:
        metadata_cols = ["form_id", "identifier", "created_at", "datapoint_id", "submitter", "name", "parent", "parent_name", "administration", "geo"]
    
    # Build a list of (question_id, label, order) tuples
    question_columns = []
    
    for fq in seed_questions:
        for q in fq["questions"]:
            qid = q[MIS_QUESTION_ID_COL]
            qlabel = q.get(MIS_QUESTION_LABEL_COL, qid)
            # Use the actual mis_question_order value from the mapping
            qorder = q.get(MIS_QUESTION_ORDER_COL, float("inf"))
            if pd.notna(qlabel) and qlabel != "":
                question_columns.append((qid, qlabel, qorder))
    
    # Sort by mis_question_order (numerical value)
    question_columns.sort(key=lambda x: x[2] if pd.notna(x[2]) else float("inf"))
    
    # Build ordered list: metadata columns first, then question columns (by label)
    ordered_columns = []
    for col in metadata_cols:
        if col not in ordered_columns:
            ordered_columns.append(col)
    
    for qid, qlabel, _ in question_columns:
        if qlabel not in ordered_columns:
            ordered_columns.append(qlabel)
    
    return ordered_columns


def build_ordered_column_list_by_id(
    seed_questions: List[dict],
    metadata_cols: List[str] = None
) -> List[str]:
    """
    Build an ordered list of column names using question IDs (not labels).
    Used when use_human_readable = False.
    
    Args:
        seed_questions: List of form mappings with questions sorted by order
        metadata_cols: List of metadata column names to place first (optional)
    
    Returns:
        Ordered list of column names (question IDs)
    """
    if metadata_cols is None:
        metadata_cols = ["form_id", "identifier", "created_at", "datapoint_id", "submitter", "name", "parent", "parent_name", "administration", "geo"]
    
    question_columns = []
    for fq in seed_questions:
        for q in fq["questions"]:
            qid = q[MIS_QUESTION_ID_COL]
            qorder = q.get(MIS_QUESTION_ORDER_COL, float("inf"))
            question_columns.append((qid, qorder))
    
    # Sort by mis_question_order
    question_columns.sort(key=lambda x: x[1] if pd.notna(x[1]) else float("inf"))
    
    # Build ordered list with question IDs
    ordered_columns = list(metadata_cols)
    for qid, _ in question_columns:
        if str(qid) not in ordered_columns:
            ordered_columns.append(str(qid))
    
    return ordered_columns

In [ ]:
# Configuration for output format
# False = question IDs (for seeder), True = human-readable labels (for manual review)
use_human_readable = False

# Output directory based on configuration
OUTPUT_DIR = "./output/mis_data_entry" if use_human_readable else os.path.join(FLOW_SOURCE_DIR, "data")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

flow_form_id = '8520967'
flow_ids = {
    "8520967": 1749634736797, # WTP
    "17260923": 1748903240763, # WWTP
    "27040920": 1749611049520, # SPS (Pump Station)
    "1520924": 1749623934933, # EPS Water quality
    "5530933": 1749623934933, # EPS Project Construction
    "2490944": 1749621221728, # RWS
}

# Mapping of Flow form IDs to their MIS child form IDs
flow_mis_child_forms = {
    "8520967": [
        1749640508297,  # WAF Water Treatment Plant - Quick Monitoring
        1749652214711,  # WAF Water Treatment Plant - Monitoring
    ],
    "17260923": [
        1748905550055,  # WWTP - Quick Monitoring
        1748918946591,  # WWTP - Monitoring
    ],
    "27040920": [
        1749611905372,  # SPS - Quick Monitoring
        1749627302948,  # SPS - Monitoring
    ],
    "1520924": [
        1749632545233,  # EPS - Water Quality Monitoring
    ],
    "5530933": [
        1749624452908,  # EPS - Project Construction Monitoring
    ],
    "2490944": [
        1749621962296,  # RWS - Quick Monitoring
        1749631041125,  # RWS - Monitoring
    ]
}

mis_form_id = flow_ids[flow_form_id]  # Get MIS form ID first
data_df = load_data_file(flow_form_id)

# Get expected child form IDs for this flow form
expected_child_form_ids = flow_mis_child_forms.get(flow_form_id, [])
seed_questions = load_question_mappings(
    flow_form_id,
    mis_form_id,
    child_form_ids=expected_child_form_ids
)

In [ ]:
parent_data = []
child_data = []
caddisfly_data = []
question_id_to_label = {}  # Store question ID to label mappings

"""
parent data is from mis parent form
child data is from mis child forms
"""
for idx, row in data_df.iterrows():
    for fq in seed_questions:
        f_qs = load_json_form(mis_form_id=str(int(fq[MIS_FORM_ID_COL])))
        meta_questions = list(filter(
            lambda  q: ((q["meta"] is True or q["meta"] == "true") and q["type"] != "geo"),
            f_qs
        ))
        meta_question_ids = [str(mq["id"]) for mq in meta_questions]
        meta_name = row["displayName"]

        questions = list(filter(lambda q: q[FLOW_QUESTION_ID_COL] in row, fq["questions"]))
        
        # Build question ID to label mapping
        for q in questions:
            qid = q[MIS_QUESTION_ID_COL]
            qlabel = q.get(MIS_QUESTION_LABEL_COL, qid)
            if pd.notna(qlabel) and qlabel != "":
                # Store both string and float versions to handle pandas' automatic conversion
                question_id_to_label[qid] = qlabel
                question_id_to_label[str(qid)] = qlabel
                # Also add version with .0 suffix for float-converted IDs
                if '.' not in str(qid):
                    question_id_to_label[f"{qid}.0"] = qlabel
            else:
                # Fallback to question ID if label not available
                question_id_to_label[qid] = qid
                question_id_to_label[str(qid)] = qid
        
        row_data = {
            q[MIS_QUESTION_ID_COL]: transform_mis_value(
                questions=f_qs,
                question_id=q[MIS_QUESTION_ID_COL],
                answer=row[q[FLOW_QUESTION_ID_COL]]
            )
            for q in questions
            if pd.notna(row[q[FLOW_QUESTION_ID_COL]])
        }

        # find row_data dict value that have caddisfly type
        caddisfly_value = next(filter(
            lambda item: isinstance(item[1], dict) and TYPE_KEY in item[1] and item[1][TYPE_KEY] == CADDISFLY_TYPE,
            row_data.items()
        ), None)
        if caddisfly_value:
            caddisfly_item = {
                "flow_form_id": flow_form_id,
                "flow_datapoint_id": row["datapoint_id"],
                "mis_form_id": fq[MIS_FORM_ID_COL],
                "mis_question_id": caddisfly_value[0],
                **caddisfly_value[1]   
            }
            # Extract 'result' as new fields for CSV
            result_list = caddisfly_value[1].get(CADDISFLY_RESULT, [])
            for res in result_list:
                c_field_key = f"{res.get(CADDISFLY_NAME_KEY)} {res.get(CADDISFLY_UNIT_KEY)}"
                caddisfly_item[c_field_key] = res.get(VALUE_KEY)
            # replace row_data caddisfly value with only the last result value
            row_data[caddisfly_value[0]] = None
            if result_list:
                row_data[caddisfly_value[0]] = result_list[-1].get(VALUE_KEY)
            # remove full caddisfly data from caddisfly_item
            del caddisfly_item[CADDISFLY_RESULT]
            caddisfly_data.append(caddisfly_item)

        if len(meta_question_ids) > 0:
            meta_available = list(filter(
                lambda m: m,
                [
                    row_data[mq] if mq in row_data else None
                    for mq in meta_question_ids
                ]
            ))
            if len(meta_available) == 0:
                meta_available = [row["displayName"]]
            # meta_available = [row["displayName"]] + meta_available
            meta_name = " - ".join(meta_available)

        geo_value = None
        administration_value = None
        adm_question = next(filter(
            lambda q: q["type"] == "administration",
            f_qs
        ), None)
        if adm_question and str(adm_question["id"]) in row_data:
            administration_value = row_data[str(adm_question["id"])]
        geo_question = next(filter(
            lambda q: q["type"] == "geo",
            f_qs
        ), None)
        if geo_question and str(geo_question["id"]) in row_data:
            geo_value = row_data[str(geo_question["id"])]
        
        if fq[MIS_FORM_ID_COL] == mis_form_id:
            parent_data.append({
                "form_id": mis_form_id,
                "identifier": row["identifier"],
                "created_at": row["createdAt"],
                "datapoint_id": row["datapoint_id"],
                "submitter": row.get("submitter", None),
                "name": meta_name,
                "administration": administration_value,
                "geo": geo_value,
                **row_data
            })
            # Store the parent name for child records to use
            parent_name_for_child = meta_name
        else:
            # Filter out empty rows
            if any(v not in [None, ""] for v in row_data.values()):
                child_data.append({
                    "form_id": fq[MIS_FORM_ID_COL],
                    "identifier": row["identifier"],
                    "created_at": row["createdAt"],
                    "datapoint_id": row["datapoint_id"],
                    "submitter": row.get("submitter", None),
                    "name": meta_name,
                    "parent_name": parent_name_for_child,  # Add parent_name for mapping
                    **row_data
                })

print(f"Built question ID to label mapping with {len(question_id_to_label)} entries")

In [ ]:
parent_data_df = pd.DataFrame(parent_data)
child_data_df = pd.DataFrame(child_data)

In [ ]:
# parent_data_df = parent_data_df.drop_duplicates(subset=["name"])
# Filter out parent records with administration value as empty and geo value as empty
parent_data_df = parent_data_df[
    (parent_data_df["administration"].notna() & (parent_data_df["administration"] != "")) &
    (parent_data_df["geo"].notna() & (parent_data_df["geo"] != ""))
]

# Only rename columns to labels if use_human_readable is True
if use_human_readable:
    sorted_question_id_to_label = build_sorted_column_mapping(
        question_id_to_label, seed_questions
    )
    parent_data_df = parent_data_df.rename(columns=sorted_question_id_to_label)
    
    # Build ordered column list and reorder (uses labels)
    parent_ordered_columns = build_ordered_column_list(
        seed_questions,
        metadata_cols=["form_id", "identifier", "created_at", "datapoint_id",
                       "submitter", "name", "administration", "geo"]
    )
else:
    # Keep question IDs but still reorder by mis_question_order
    parent_ordered_columns = build_ordered_column_list_by_id(
        seed_questions,
        metadata_cols=["form_id", "identifier", "created_at", "datapoint_id",
                       "submitter", "name", "administration", "geo"]
    )

# Only include columns that actually exist in the DataFrame
parent_ordered_columns = [col for col in parent_ordered_columns if col in parent_data_df.columns]
# Add any remaining columns not in our ordered list (to preserve all data)
remaining_columns = [col for col in parent_data_df.columns if col not in parent_ordered_columns]
parent_data_df = parent_data_df[parent_ordered_columns + remaining_columns]

# Save parent data to OUTPUT_DIR
parent_data_df.to_csv(
    os.path.join(OUTPUT_DIR, f"{flow_form_id}_parent_data.csv"),
    index=False
)
print("total parent records:", len(parent_data_df))
print(f"parent data saved to: {OUTPUT_DIR}")

# Filter child_data_df based on datapoint_id in parent_data_df
# Handle empty child_data case
print(f"total child records before filtering: {len(child_data)}")
if len(child_data) > 0:
    # Group child data by form_id to create separate CSV files per child form
    child_data_by_form = {}
    for record in child_data:
        form_id = record.get("form_id")
        if form_id not in child_data_by_form:
            child_data_by_form[form_id] = []
        child_data_by_form[form_id].append(record)
    
    print(f"Found {len(child_data_by_form)} child form(s) with data")
    
    # Group parent_data_df by 'name' and create name -> datapoint_id mapping
    parent_name_mapping = parent_data_df.groupby("name")["datapoint_id"].first().to_dict()
    print(f"unique parent names: {len(parent_name_mapping)}")
    
    # Process each child form separately
    for child_form_id, child_form_data in child_data_by_form.items():
        print(f"\n--- Processing child form_id: {child_form_id} ---")
        
        # Create DataFrame for this child form
        child_form_df = pd.DataFrame(child_form_data)
        
        # Filter to only include datapoint_ids that exist in parent_data_df
        child_form_df = child_form_df[
            child_form_df["datapoint_id"].isin(parent_data_df["datapoint_id"])
        ]
        
        if len(child_form_df) == 0:
            print(f"  No data rows for child form {child_form_id} after filtering")
            continue
        
        # Add 'parent' field using the parent_name mapping
        child_form_df["parent"] = child_form_df["parent_name"].map(parent_name_mapping)
        
        # Drop the parent_name column as it's only needed for mapping
        child_form_df = child_form_df.drop(columns=["parent_name"])
        
        # Verify: check if any child records didn't get a parent assigned
        orphaned_children = child_form_df[child_form_df["parent"].isna()]
        if len(orphaned_children) > 0:
            print(f"  warning: {len(orphaned_children)} child records without parent match")
        
        # Get seed_questions for only this child form (for proper column ordering)
        child_form_seed_questions = [
            fq for fq in seed_questions 
            if fq[MIS_FORM_ID_COL] == child_form_id
        ]
        
        # Apply conditional column renaming and ordering
        if child_form_seed_questions:
            if use_human_readable:
                # Build label mapping and rename columns
                child_form_question_mapping = {}
                for q in child_form_seed_questions[0]["questions"]:
                    qid = q[MIS_QUESTION_ID_COL]
                    qlabel = q.get(MIS_QUESTION_LABEL_COL, qid)
                    if pd.notna(qlabel) and qlabel != "":
                        child_form_question_mapping[qid] = qlabel
                        child_form_question_mapping[str(qid)] = qlabel
                        if '.' not in str(qid):
                            child_form_question_mapping[f"{qid}.0"] = qlabel
                
                child_sorted_mapping = build_sorted_column_mapping(
                    child_form_question_mapping, child_form_seed_questions
                )
                child_form_df = child_form_df.rename(columns=child_sorted_mapping)
                
                child_ordered_columns = build_ordered_column_list(
                    child_form_seed_questions,
                    metadata_cols=["form_id", "identifier", "created_at", "datapoint_id",
                                  "submitter", "name", "parent"]
                )
            else:
                # Keep question IDs but reorder by mis_question_order
                child_ordered_columns = build_ordered_column_list_by_id(
                    child_form_seed_questions,
                    metadata_cols=["form_id", "identifier", "created_at", "datapoint_id",
                                  "submitter", "name", "parent"]
                )
        else:
            # Fallback if no seed_questions found for this form
            child_ordered_columns = ["form_id", "identifier", "created_at", "datapoint_id", "submitter", "name", "parent"]
        
        # Only include columns that actually exist in the DataFrame
        child_ordered_columns = [col for col in child_ordered_columns if col in child_form_df.columns]
        # Add any remaining columns not in our ordered list (to preserve all data)
        remaining_columns = [col for col in child_form_df.columns if col not in child_ordered_columns]
        child_form_df = child_form_df[child_ordered_columns + remaining_columns]
        print(f"  total child records after filtering: {len(child_form_df)}")
        
        print(f"  Columns: {child_form_df.columns.tolist()}")
        
        # Save to a separate CSV file per child form
        child_output_filename = f"{flow_form_id}_child_data_{int(child_form_id)}.csv"
        child_form_df.to_csv(
            os.path.join(OUTPUT_DIR, child_output_filename),
            index=False
        )
        print(f"  Saved {len(child_form_df)} rows to {child_output_filename}")
    
    print(f"\nChild data saved in {len(child_data_by_form)} separate file(s) (one per child form)")
else:
    print("No child data to save")

if len(caddisfly_data) > 0:
    # convert caddisfly_data to CSV file
    caddisfly_data_df = pd.DataFrame(caddisfly_data)
    caddisfly_data_df.to_csv(
        os.path.join(FLOW_SOURCE_DIR, "caddisfly", f"{flow_form_id}_caddisfly_data.csv"),
        index=False
    )
    print("caddisfly data saved")